# 04b. 피처 엔지니어링 — KURE-v1 SBERT 기반 seed 확장

## 왜 fastText에서 SBERT로 바꾸는가

기존 [04_피처엔지니어링.ipynb](04_피처엔지니어링.ipynb)는 DART 보고서 코퍼스로 fastText를 **자체 학습**해 seed 30개의 유사어를 뽑았습니다. 결과 품질이 낮았는데, 원인은 형태소 분석이 아니라 fastText 자체의 한계입니다:

1. **분포 학습 ≠ 의미 학습**. fastText는 "함께 자주 나타난 단어"를 가깝게 만듭니다. 그래서 `탄소`의 1위 유사어가 `로드맵`(탄소중립 로드맵 → 늘 같이 등장)으로 잡힙니다.
2. **subword가 만드는 가짜 유사도**. 글자 n-gram을 공유하면 의미가 달라도 가까워집니다. `탄소` vs `탄소강`(금속 소재)이 상위에 들어옵니다.
3. **작은 말뭉치**. DART 사업보고서 ~380개는 ESG 희귀어를 충분히 학습하기에 작아서, 형태소 조각이 잡음으로 들어옵니다.

**해결책**: 외부 대규모 한국어 데이터로 이미 의미를 학습한 사전학습 임베딩 모델을 쓴다. 이 노트북은 [nlpai-lab/KURE-v1](https://huggingface.co/nlpai-lab/KURE-v1) (고려대 NLP&AI Lab, BAAI/bge-m3를 한국어 query-document 200만 쌍으로 추가 학습한 retrieval 임베딩)로 seed 단어와 후보 단어를 인코딩한 뒤, **단어 벡터 간 코사인 유사도**로 후보를 뽑습니다.

## 핵심 설계 메모

- **비교 단위는 "단어 ↔ 단어"** 입니다. 문장 임베딩 모델 이름에 속아 문장 쌍 비교로 빠지지 않는다.
- **후보 풀**: `dart_preprocessed.csv`의 `kiwi_text` 컬럼에서 모은 고유 토큰. 보고서에 실제 등장한 단어만 후보가 된다.
- **모델 사용법**: `SentenceTransformer.encode([word1, word2, ...])` → 단어당 1024차원 벡터 1개. fastText처럼 학습/`most_similar()` 호출 없음. 프리픽스도 붙이지 않는다(KURE-v1 모델 카드 기준, 단어-단어 대칭 유사도 용도).
- **출력 스키마는 기존 fastText 산출물과 동일** — `dimension, seed_term, rank_within_seed, candidate, similarity, source_query_term`. 다운스트림 LLM 검토 파이프라인([scripts/llm_filter_candidates.py](../scripts/llm_filter_candidates.py), `00_기타.ipynb`)을 그대로 재사용한다.

## 확정된 하이퍼파라미터

| 항목 | 값 | 근거 |
|---|---|---|
| `USE_QUERY_PREFIX` | `False` | KURE-v1 모델 카드 — prefix 미언급, 단어-단어 대칭 유사도에 부착 불필요 |
| `MIN_DOC_FREQ` | `2` | df=1 단독 출현 ~13k 토큰은 회사 고유어/오탈자 비중이 큼. df≥2면 vocab 25% 감소·token 발생량은 거의 유지 |
| `TOPK_PER_SEED` | `300` | fastText 13k행 규모와 균형. 30 seed × 300 = 최대 9k행 |
| `EXCLUDE_COMPANY_NAMES` | `True` | 03 전처리에서 대체로 줄었지만 완전 제거 아님 (`회사이름`과 정확히 같은 토큰이 일부 문서에 잔존) |
| `THETA_GRID` | `[0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]` | SBERT 점수는 fastText보다 한 단계 위에 분포 |

## Colab에서 실행하기 — 준비 사항

### 업로드해야 할 파일 (3개)
이 노트북은 로컬 프로젝트의 다음 3개 파일을 입력으로 씁니다:

1. `data/processed/dart_preprocessed.csv` (~180 MB) — 03 전처리 산출물, `kiwi_text` 컬럼 사용
2. `data/raw/seed_dictionary.csv` (~6 KB) — 30개 seed (E/S/G × 10) + pattern
3. `data/raw/stopwords_ko_esg.txt` (~250 B) — 한국어 ESG 불용어

### 업로드 방법 (둘 중 택1)

- **(권장) Google Drive 마운트**: `dart_preprocessed.csv`가 크므로 매 세션 다시 올리지 않도록 Drive에 두는 게 편하다. 아래 셀의 `MOUNT_DRIVE=True`로 두고, 본인의 Drive 폴더 구조에 맞게 `INPUT_DIR` 경로만 수정.
- **수동 업로드**: `MOUNT_DRIVE=False`로 두면 `files.upload()` 다이얼로그가 뜬다 (180 MB는 시간이 좀 걸린다).

### GPU 권장
KURE-v1(1024차원)은 CPU에서도 돌지만, 후보 ~40k 단어를 인코딩하려면 T4 GPU(Colab 무료)로 1~2분이면 끝난다. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 바꿔주세요. CPU에서는 10~30분 걸릴 수 있습니다.

### 출력
`/content/outputs/`(Colab) 또는 `data/processed/feature_engineering/`(로컬)에 다음이 저장되고, Colab에서는 마지막 셀에서 자동 다운로드된다:
- `sbert_seed_candidates.csv` — seed × 후보 단어 × 유사도 (rank top 300/seed)
- `sbert_threshold_summary.csv` — θ 임계치별 후보 수와 잡음 비율
- `expanded_dictionary_review_template.csv` — 사람/LLM이 keep/drop 채울 검토 템플릿
- `sbert_similarity_histogram.png` — seed-후보 유사도 분포 (참고용)

In [ ]:
# === Colab 환경 셋업 ===
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # torch는 Colab에 미리 설치되어 있음. nbformat 호환을 위해 잠금 안 함.
    !pip install -q sentence-transformers kiwipiepy

import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

# GPU 감지
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ GPU 미감지. 후보 ~40k 단어 인코딩이 CPU에서는 10~30분 걸릴 수 있습니다.')
    print('   런타임 → 런타임 유형 변경 → GPU(T4) 권장.')

# === 경로 설정 ===
MOUNT_DRIVE = True   # Colab에서 False면 수동 업로드

if IN_COLAB:
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        # ★ 본인 Drive 구조에 맞게 수정 ★
        INPUT_DIR = Path('/content/drive/MyDrive/esg-dart-project/data')
        PREPROCESSED_CSV = INPUT_DIR / 'processed' / 'dart_preprocessed.csv'
        SEED_DICTIONARY_CSV = INPUT_DIR / 'raw' / 'seed_dictionary.csv'
        STOPWORDS_PATH = INPUT_DIR / 'raw' / 'stopwords_ko_esg.txt'
    else:
        from google.colab import files
        print('3개 파일을 업로드하세요: dart_preprocessed.csv, seed_dictionary.csv, stopwords_ko_esg.txt')
        files.upload()  # 결과는 /content/ 에 저장됨
        upload_dir = Path('/content')
        PREPROCESSED_CSV = upload_dir / 'dart_preprocessed.csv'
        SEED_DICTIONARY_CSV = upload_dir / 'seed_dictionary.csv'
        STOPWORDS_PATH = upload_dir / 'stopwords_ko_esg.txt'
    OUTPUT_DIR = Path('/content/outputs')
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    PREPROCESSED_CSV = PROJECT_ROOT / 'data' / 'processed' / 'dart_preprocessed.csv'
    SEED_DICTIONARY_CSV = PROJECT_ROOT / 'data' / 'raw' / 'seed_dictionary.csv'
    STOPWORDS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'stopwords_ko_esg.txt'
    OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'feature_engineering'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED_CANDIDATES_CSV = OUTPUT_DIR / 'sbert_seed_candidates.csv'
THRESHOLD_SUMMARY_CSV = OUTPUT_DIR / 'sbert_threshold_summary.csv'
REVIEW_TEMPLATE_CSV = OUTPUT_DIR / 'expanded_dictionary_review_template.csv'
HIST_PNG = OUTPUT_DIR / 'sbert_similarity_histogram.png'

for p in [PREPROCESSED_CSV, SEED_DICTIONARY_CSV, STOPWORDS_PATH]:
    assert p.exists(), f'필수 입력 없음: {p}'
    print(f'  ✓ {p}')

print(f'\n출력 폴더: {OUTPUT_DIR}')

## 1. 입력 데이터 로딩

03 전처리 노트북이 만든 `kiwi_text` 컬럼은 다음 단계를 이미 거친 상태다:

- 로마숫자 섹션 헤더 제거
- 한국어/영문 외 문자 제거 + NFKC 정규화 + lowercase (`clean_text`)
- Kiwi 형태소 분석 (`text_raw` 기준 — `clean_text`는 단어 경계가 깨짐)
- 불용어 + 회사명/계열사명 제거
- POS 필터: 명사(N*) · 용언(V*) · 영문(SL)만 보존

이 노트북은 추가 형태소 분석을 하지 않고 `kiwi_text`의 공백 구분 토큰을 그대로 가져다 쓴다.

In [ ]:
df = pd.read_csv(PREPROCESSED_CSV)
seed_dictionary = pd.read_csv(SEED_DICTIONARY_CSV)

with STOPWORDS_PATH.open('r', encoding='utf-8') as f:
    stopwords = {
        unicodedata.normalize('NFKC', line.strip()).lower()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }
ROMAN_STOPWORDS = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii'}
stopwords |= ROMAN_STOPWORDS

df['kiwi_text'] = df['kiwi_text'].fillna('').astype(str)

print(f'전처리 입력 행 수 (회사-연도): {len(df)}')
print(f'kiwi_text 비어있지 않은 행: {(df["kiwi_text"].str.strip() != "").sum()}')
print(f'고유 회사: {df["회사이름"].nunique()}개')
print(f'seed: {len(seed_dictionary)}개 (E/S/G 각 10)')
print(f'불용어: {len(stopwords)}개')
df[['회사이름', 'esg_year']].head()

## 2. 후보 어휘 풀(candidate pool) 구성

후보 풀은 `kiwi_text`에서 모은 **고유 토큰**이다. 단어 ↔ 단어 비교이므로 보고서에 실제 등장한 단어만 후보가 된다.

### 필터 단계

1. 정규식 `^[가-힣A-Za-z]+$` 통과 토큰만 보존 (숫자/기호/혼합 제외)
2. 한 글자 토큰 제외 (`len(token) > 1`)
3. 불용어 제외 (`stopwords_ko_esg.txt` + 로마숫자 i~xii)
4. 한글+영문 혼합 토큰 제외 (`[가-힣]`와 `[A-Za-z]`가 둘 다 포함된 경우)
5. **회사명 토큰 제외** — 03 전처리에서 대체로 줄었지만 완전 제거는 아님. `회사이름`과 정확히 같은 토큰이 일부 문서에 잔존하므로 후보 단계에서 한 번 더 자른다 (사전 검증: 89개 회사명이 247개 문서에 잔존).
6. **`MIN_DOC_FREQ=2` 컷오프** — 단일 문서에서만 등장하는 토큰(`df=1`)은 회사 고유 용어/오탈자 비중이 크다. df≥1/2/3/5/10의 vocab 크기를 같이 출력하여 향후 컷오프 조정 근거를 보존한다.

In [ ]:
TOKEN_RE = re.compile(r'^[가-힣A-Za-z]+$')
MIN_DOC_FREQ = 2  # 변경 시 후보 풀 크기가 바뀜


def normalize_word(word: object) -> str:
    return unicodedata.normalize('NFKC', str(word)).lower().strip()


def passes_token_filter(tok: str) -> bool:
    if not TOKEN_RE.fullmatch(tok):
        return False
    if len(tok) <= 1:
        return False
    if tok in stopwords:
        return False
    if re.search(r'[가-힣]', tok) and re.search(r'[A-Za-z]', tok):
        return False
    return True


# 회사명 토큰 집합 (정확히 같은 토큰만 제거 — 부분 매칭은 의미 손실이 크다)
company_name_terms = {normalize_word(n) for n in df['회사이름'].dropna() if str(n).strip()}
print(f'회사명 집합: {len(company_name_terms)}개')

# 토큰별 회사-연도 문서 등장 수 (document frequency)
doc_freq: Counter = Counter()
company_token_doc_count: Counter = Counter()  # 보고용 — 회사명이 잔존한 빈도

for kiwi_text in df['kiwi_text']:
    seen_in_doc: set = set()
    for tok in kiwi_text.split():
        tok = normalize_word(tok)
        if passes_token_filter(tok):
            seen_in_doc.add(tok)
    for tok in seen_in_doc:
        if tok in company_name_terms:
            company_token_doc_count[tok] += 1
        else:
            doc_freq[tok] += 1

print(f'\n[회사명 잔존 검증] 회사명과 정확히 같은 토큰이 잔존한 문서:')
print(f'  · 회사 {len(company_token_doc_count)}개, 누적 문서 {sum(company_token_doc_count.values())}개')
print(f'  → 모두 후보 풀에서 제외됨')

print(f'\n필터 통과 + 회사명 제외 후 고유 토큰 (df≥1): {len(doc_freq):,}')

# MIN_DOC_FREQ 민감도 비교표
sensitivity = pd.DataFrame({
    'min_doc_freq': [1, 2, 3, 5, 10],
    'vocab_size': [sum(1 for c in doc_freq.values() if c >= k) for k in [1, 2, 3, 5, 10]],
})
sensitivity['drop_vs_df1'] = (sensitivity['vocab_size'].iloc[0] - sensitivity['vocab_size']).astype(int)
sensitivity['drop_pct'] = (sensitivity['drop_vs_df1'] / sensitivity['vocab_size'].iloc[0] * 100).round(1)
print('\nmin_doc_freq 민감도:')
print(sensitivity.to_string(index=False))
print(f'\n→ 기본값 MIN_DOC_FREQ={MIN_DOC_FREQ} 사용')

candidate_pool = sorted([tok for tok, c in doc_freq.items() if c >= MIN_DOC_FREQ])
print(f'\n최종 후보 풀: {len(candidate_pool):,}개')
print('\n예시 — doc_freq 상위 20개 (자주 등장하는 일반 토큰):')
for tok, c in sorted(doc_freq.items(), key=lambda x: -x[1])[:20]:
    print(f'  {tok:>15s}  doc_freq={c}')

## 3. seed → query term 분해

`seed_term` (예: `재생에너지`)과 `pattern` (예: `재생에너지|renewable energy`)을 Kiwi로 분해해 query term 목록을 만든다. SBERT는 어떤 입력이든 인코딩할 수 있으므로 fastText 버전에 있던 "vocab에 단어가 존재하는가" 필터는 빼도 된다 — 단어 자체가 의미 있는 표현이기만 하면 된다.

분해 정책 (기존 04 노트북과 동일):

- `pattern`의 `|`로 분리한 각 표현을 별도 처리
- 한글/영문 외 문자를 제거한 컴팩트 형태도 추가 (`net zero` → `netzero`)
- `[^가-힣A-Za-z]+` 기준으로 쪼갠 partial도 추가
- Kiwi 형태소 분석 결과 중 `N*` / `V*` / `SL` (영문)만 보존
- 한 글자 / 불용어 / 비문자 토큰 제외

> **모델 프리픽스 메모**: KURE-v1 모델 카드는 `encode()`에 raw 문장을 그대로 넣는 예시만 보여준다. retrieval에서 query/passage 비대칭 인스트럭션이 일반적이지만 여기서는 단어-단어 대칭 유사도이므로 **프리픽스 미부착**으로 진행한다. e5 계열을 폴백으로 쓸 때만 `"query: "` 분기 처리가 필요하다.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
KEEP_POS_PREFIXES = ('N', 'V')
KEEP_POS_TAGS = {'SL'}


def tokenize_seed_phrase(phrase: object) -> list[str]:
    if not isinstance(phrase, str) or not phrase.strip():
        return []
    phrase = normalize_word(phrase)
    out: set[str] = set()
    compact = re.sub(r'[^가-힣A-Za-z]+', '', phrase)
    if len(compact) > 1:
        out.add(compact)
    for part in re.split(r'[^가-힣A-Za-z]+', phrase):
        if len(part) > 1 and TOKEN_RE.fullmatch(part):
            out.add(part)
    for token in kiwi.tokenize(phrase):
        word = normalize_word(token.form)
        if len(word) <= 1 or not TOKEN_RE.fullmatch(word):
            continue
        if not (token.tag.startswith(KEEP_POS_PREFIXES) or token.tag in KEEP_POS_TAGS):
            continue
        out.add(word)
    return sorted(out)


def seed_query_terms(row: pd.Series) -> list[str]:
    phrases = [row['seed_term']]
    if isinstance(row.get('pattern'), str):
        phrases.extend(row['pattern'].split('|'))
    terms: list[str] = []
    for phrase in phrases:
        terms.extend(tokenize_seed_phrase(phrase))
    return sorted({t for t in terms if t not in stopwords})


seed_dictionary['query_terms'] = seed_dictionary.apply(seed_query_terms, axis=1)
seed_dictionary['query_count'] = seed_dictionary['query_terms'].map(len)

# 모든 seed query term의 union — 후보 풀에서 제외할 집합
ALL_SEED_TERMS: set[str] = set()
for terms in seed_dictionary['query_terms']:
    ALL_SEED_TERMS.update(terms)

pd.set_option('display.max_colwidth', 80)
print('seed별 query term 분해 결과:')
print(seed_dictionary[['dimension', 'seed_term', 'pattern', 'query_count', 'query_terms']].to_string(index=False))
pd.reset_option('display.max_colwidth')

print(f'\n총 query term (union, 중복 제거): {len(ALL_SEED_TERMS)}개')

# 후보 풀에서 seed query term 자기 자신 제거
candidate_pool_for_seeds = [c for c in candidate_pool if c not in ALL_SEED_TERMS]
print(f'seed 자기 자신 제외 후 최종 후보 풀: {len(candidate_pool_for_seeds):,}개')

## 4. KURE-v1 모델 로드 & 임베딩

`nlpai-lab/KURE-v1`은 BAAI/bge-m3 위에 한국어 query-document 200만 쌍을 추가 학습한 retrieval 임베딩 모델 (1024차원, max seq 8192). ESG 보고서의 법률/금융 문어체와 결이 비슷해 도메인 적합도가 좋다.

**왜 단어를 문장 인코더에 넣어도 되는가**: sentence-transformers의 `encode()`는 입력 길이에 무관하게 문맥화 표현을 평균/풀링해서 단일 벡터를 만든다. 단어 1개를 넣으면 "그 단어만 있는 짧은 시퀀스"의 풀링 벡터가 나온다. 이는 BERT류 모델에서 단어 임베딩을 얻는 표준적인 방식이다.

**구현 디테일**:
- `normalize_embeddings=True` — 결과 벡터가 단위 노름(`||v||=1`)이라 코사인 유사도가 단순 내적으로 떨어진다.
- `batch_size=64` (GPU) / `16` (CPU)
- `convert_to_tensor=True` — 다음 단계의 `util.cos_sim`이 텐서 입력 기대

모델 첫 로드 시 ~2 GB 다운로드 (Colab 캐시에 저장됨, 같은 세션에서 재실행 시 즉시).

In [ ]:
MODEL_NAME = 'nlpai-lab/KURE-v1'
USE_QUERY_PREFIX = False  # KURE-v1: 단어 단위 대칭 유사도 → 프리픽스 미부착
BATCH_SIZE = 64 if DEVICE == 'cuda' else 16

print(f'모델 로드: {MODEL_NAME} (device={DEVICE})')
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f'  · embedding dim: {model.get_sentence_embedding_dimension()}')
print(f'  · max seq length: {model.max_seq_length}')


def to_input(words: list[str]) -> list[str]:
    # e5 계열 폴백을 쓸 경우만 'query: ' 분기. KURE-v1은 그대로.
    if USE_QUERY_PREFIX:
        return [f'query: {w}' for w in words]
    return words


# 1) seed query term 임베딩
all_seed_query_terms = sorted(ALL_SEED_TERMS)
print(f'\n[1/2] seed query term 인코딩: {len(all_seed_query_terms)}개')
seed_emb = model.encode(
    to_input(all_seed_query_terms),
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_tensor=True,
)
print(f'  → shape: {tuple(seed_emb.shape)}')

# 2) 후보 어휘 임베딩
print(f'\n[2/2] 후보 어휘 인코딩: {len(candidate_pool_for_seeds):,}개')
cand_emb = model.encode(
    to_input(candidate_pool_for_seeds),
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_tensor=True,
)
print(f'  → shape: {tuple(cand_emb.shape)}')

# seed query term → 인덱스 매핑 (이후 lookup용)
seed_term_to_idx = {term: i for i, term in enumerate(all_seed_query_terms)}
print(f'\n인코딩 완료. seed_term_to_idx 사이즈: {len(seed_term_to_idx)}')

## 5. seed별 top-K 후보 추출

각 seed에 대해, 해당 seed의 모든 query term을 후보 어휘 전체와 코사인 유사도 비교한다. 한 seed가 여러 query term을 가지면(예: `넷제로` → `넷제로 / netzero / net`), 각 query term의 유사도 중 **최대값**을 그 후보의 점수로 쓴다 (= 기존 04 노트북과 동일한 best-of policy). 어떤 query term이 끌어올렸는지는 `source_query_term`에 남긴다.

`TOPK_PER_SEED = 300`. fastText 버전(query term당 200, query term 83개 → 13,137행 산출)과 규모를 맞추되 약간 더 넓게 잡는다 (30 seed × 300 = 최대 9k행).

코사인 유사도 계산은 normalize된 벡터의 행렬곱 한 번이면 끝난다 — `util.cos_sim(seed_emb, cand_emb)`.

In [ ]:
TOPK_PER_SEED = 300

# 전체 유사도 행렬: [N_seed_query_terms, N_cands]
# 행 = all_seed_query_terms, 열 = candidate_pool_for_seeds
sim_matrix = util.cos_sim(seed_emb, cand_emb)
print(f'유사도 행렬: {tuple(sim_matrix.shape)}')

rows = []
for seed_row in seed_dictionary.itertuples(index=False):
    query_terms = list(seed_row.query_terms)
    if not query_terms:
        continue
    # 이 seed의 query term 중 인코딩된 것만 추리고 순서 보존
    kept = [(seed_term_to_idx[t], t) for t in query_terms if t in seed_term_to_idx]
    if not kept:
        continue
    q_idx = [i for i, _ in kept]
    q_names = [n for _, n in kept]

    sub = sim_matrix[q_idx]                       # [n_q, N_cands]
    best_sim, best_q_row = sub.max(dim=0)         # [N_cands], [N_cands]

    topk = torch.topk(best_sim, k=min(TOPK_PER_SEED, best_sim.shape[0]))
    for rank, (cand_i, sim_val) in enumerate(
        zip(topk.indices.tolist(), topk.values.tolist()), start=1
    ):
        q_term = q_names[best_q_row[cand_i].item()]
        rows.append({
            'dimension': seed_row.dimension,
            'seed_term': seed_row.seed_term,
            'rank_within_seed': rank,
            'candidate': candidate_pool_for_seeds[cand_i],
            'similarity': float(sim_val),
            'source_query_term': q_term,
        })

seed_candidates = pd.DataFrame(rows)
print(f'\nseed-candidate pair 수: {len(seed_candidates):,}')
print(f'고유 후보 단어 수: {seed_candidates["candidate"].nunique():,}')

print('\ndimension별 후보 통계:')
print(seed_candidates.groupby('dimension').agg(
    pairs=('candidate', 'size'),
    unique_candidates=('candidate', 'nunique'),
    mean_sim=('similarity', 'mean'),
    median_sim=('similarity', 'median'),
    p95_sim=('similarity', lambda s: s.quantile(0.95)),
    max_sim=('similarity', 'max'),
).round(3).to_string())

print('\n예시 — "탄소" seed의 상위 20개:')
print(seed_candidates.query('seed_term == "탄소"').head(20).to_string(index=False))

print('\n예시 — "이사회" seed의 상위 20개:')
print(seed_candidates.query('seed_term == "이사회"').head(20).to_string(index=False))

## 6. 유사도 분포 — SBERT는 fastText와 다르다

KURE-v1은 사전학습된 임베딩이라 fastText보다 유사도 점수가 전반적으로 높고 빽빽한 분포를 가진다. fastText에서 θ=0.60이 "유의미 컷"이었다고 해서 SBERT에서도 같은 의미라고 가정하면 안 된다.

히스토그램과 percentile을 먼저 보고, 다음 셀에서 θ grid를 데이터에 맞게 sweep한다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# Colab 한글 폰트 (없으면 그냥 진행 — 한글이 깨져도 분포 자체는 보임)
try:
    if IN_COLAB:
        import subprocess
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=False)
    for fname in ['NanumGothic', 'AppleGothic', 'Malgun Gothic']:
        try:
            matplotlib.rc('font', family=fname)
            break
        except Exception:
            continue
    matplotlib.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (좌) 전체 유사도 분포 + percentile 표시
ax = axes[0]
ax.hist(seed_candidates['similarity'], bins=60, color='#3D7EFF', alpha=0.8)
for p in [50, 75, 90, 95, 99]:
    v = float(np.percentile(seed_candidates['similarity'], p))
    ax.axvline(v, color='gray', linestyle='--', linewidth=0.8)
    ax.text(v, ax.get_ylim()[1] * 0.95, f' p{p}={v:.3f}',
            fontsize=9, rotation=90, va='top')
ax.set_title('KURE-v1 seed-candidate cosine similarity')
ax.set_xlabel('cosine similarity')
ax.set_ylabel('count')

# (우) dimension별 분포
ax = axes[1]
for dim, color in zip(['E', 'S', 'G'], ['#2EA02E', '#FF7F0E', '#9467BD']):
    s = seed_candidates.loc[seed_candidates['dimension'] == dim, 'similarity']
    ax.hist(s, bins=40, color=color, alpha=0.5, label=f'{dim} (n={len(s)})')
ax.set_title('Dimension-wise similarity distribution')
ax.set_xlabel('cosine similarity')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout()
plt.savefig(HIST_PNG, dpi=100, bbox_inches='tight')
plt.show()
print(f'\n저장: {HIST_PNG}')

print('\nseed-candidate 유사도 percentile:')
percentiles = pd.Series({
    f'p{p}': float(np.percentile(seed_candidates['similarity'], p))
    for p in [50, 60, 70, 75, 80, 85, 90, 95, 99]
})
print(percentiles.round(4).to_string())

## 7. θ 임계치 sweep + 잡음 휴리스틱

각 θ에서:

- 유사도 ≥ θ인 (seed, candidate) 쌍만 살림
- candidate 단위로 집계 — `primary_dimension`(최대 점수가 나온 E/S/G), `seed_support_count`(매달린 seed 수), `dimension_support_count`(E/S/G 중 몇 개)
- 휴리스틱 잡음 플래그를 단다 → `top100_noise_ratio`를 보고 θ를 고른다

### 잡음 휴리스틱 (`heuristic_noise_flag`) — 기존 04와 동일

- **long_compound**: 12자 초과 한글 + ESG 핵심어가 단어 안에 없음
- **short_english**: 3자 이하 영문 (예외: `esg`, `ghg`, `iso`, `co2`)
- **english_no_esg_hint**: 4자 이상 영문이지만 ESG 힌트가 없음
- **broad_low_specificity**: 8개 이상 seed에 매달리지만 최대 유사도가 낮음 → "광범위 저특이성"

### θ grid 선정 근거

fastText 노트북은 `[0.45, 0.50, …, 0.75]`였다. SBERT(KURE-v1)는 단어 코사인 유사도 분포가 전반적으로 위쪽에 몰리므로 `[0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]`로 한 단계 시프트한다. 실제 percentile을 본 뒤 노트북에서 미세조정 가능.

In [ ]:
ESG_EN_HINTS = {
    'carbon', 'climate', 'netzero', 'renewable', 'energy', 'recycle', 'recycling',
    'waste', 'water', 'safety', 'human', 'rights', 'supply', 'compliance',
    'ethics', 'board', 'audit', 'shareholder', 'governance', 'emission', 'green',
}
CORE_KO_HINTS = {
    '탄소', '온실', '기후', '재생', '에너지', '전력', '폐기', '재활용', '폐수', '배출',
    '안전', '재해', '임직원', '직원', '근로자', '노동', '인권', '교육', '훈련', '협력', '공급망',
    '이사회', '사외', '이사', '감사', '위원회', '독립', '윤리', '준법', '부패', '주주', '의결권',
}


def noise_reason(word: str, seed_support: int, dim_support: int, max_sim: float) -> str:
    reasons = []
    if re.fullmatch(r'[가-힣]+', word) and len(word) > 12 and not any(h in word for h in CORE_KO_HINTS):
        reasons.append('long_compound')
    if re.fullmatch(r'[a-z]+', word) and len(word) <= 3 and word not in {'esg', 'ghg', 'iso', 'co2'}:
        reasons.append('short_english')
    if re.fullmatch(r'[a-z]+', word) and len(word) > 3 and not any(h in word for h in ESG_EN_HINTS):
        reasons.append('english_no_esg_hint')
    if seed_support >= 8 and dim_support >= 2 and max_sim < 0.65:
        reasons.append('broad_low_specificity')
    return '|'.join(reasons)


def aggregate_at_theta(theta: float) -> pd.DataFrame:
    sub = seed_candidates.loc[seed_candidates['similarity'] >= theta]
    if sub.empty:
        return pd.DataFrame(columns=[
            'candidate', 'primary_dimension', 'max_similarity', 'mean_similarity',
            'seed_support_count', 'dimension_support_count', 'supporting_seeds',
            'heuristic_noise_reason', 'heuristic_noise_flag',
        ])
    out = []
    for candidate, group in sub.groupby('candidate'):
        dim_scores = group.groupby('dimension')['similarity'].max().sort_values(ascending=False)
        max_sim = float(group['similarity'].max())
        seed_support = int(group[['dimension', 'seed_term']].drop_duplicates().shape[0])
        dim_support = int(group['dimension'].nunique())
        reason = noise_reason(candidate, seed_support, dim_support, max_sim)
        out.append({
            'candidate': candidate,
            'primary_dimension': dim_scores.index[0],
            'max_similarity': max_sim,
            'mean_similarity': float(group['similarity'].mean()),
            'seed_support_count': seed_support,
            'dimension_support_count': dim_support,
            'supporting_seeds': '|'.join(sorted(group['seed_term'].unique())),
            'heuristic_noise_reason': reason,
            'heuristic_noise_flag': bool(reason),
        })
    return pd.DataFrame(out).sort_values('max_similarity', ascending=False).reset_index(drop=True)


THETA_GRID = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]
TOP_REVIEW = 100

summary_rows = []
aggregated_by_theta: dict[float, pd.DataFrame] = {}
for theta in THETA_GRID:
    agg = aggregate_at_theta(theta)
    aggregated_by_theta[theta] = agg
    if agg.empty:
        summary_rows.append({
            'theta': theta, 'unique_candidates': 0,
            'top100_noise_count': 0, 'top100_noise_ratio': np.nan,
            'E_terms': 0, 'S_terms': 0, 'G_terms': 0,
        })
        continue
    top = agg.head(TOP_REVIEW)
    summary_rows.append({
        'theta': theta,
        'unique_candidates': len(agg),
        'top100_noise_count': int(top['heuristic_noise_flag'].sum()),
        'top100_noise_ratio': round(float(top['heuristic_noise_flag'].mean()), 3),
        'E_terms': int((agg['primary_dimension'] == 'E').sum()),
        'S_terms': int((agg['primary_dimension'] == 'S').sum()),
        'G_terms': int((agg['primary_dimension'] == 'G').sum()),
    })

threshold_summary = pd.DataFrame(summary_rows)
print('θ별 후보 규모와 잡음 비율:')
print(threshold_summary.to_string(index=False))

## 8. 최종 θ 선택 + 검토 템플릿 생성

위 표에서 `top100_noise_ratio`가 낮아지면서 `unique_candidates`가 너무 적어지지 않는 지점을 고른다. SBERT는 fastText 대비 θ가 한 단계 위(예: 0.65~0.75)로 이동하는 게 일반적이다.

**기본값 `FINAL_THETA = 0.65`** 로 두되, 위 표를 보고 셀에서 값을 수정한 뒤 재실행해 확정한다. 어떤 근거로 θ를 골랐는지는 아래 마크다운에 한 줄 적어두면 좋다 (예: "θ=0.70에서 top100_noise_ratio가 0.18로 떨어지고 unique 후보가 ~500개 → 검토 부담과 recall 균형").

검토 템플릿(`expanded_dictionary_review_template.csv`)은 기존 04 노트북과 동일한 형식이라 [scripts/llm_filter_candidates.py](../scripts/llm_filter_candidates.py)와 `00_기타.ipynb`가 코드 수정 없이 사용 가능하다. `term_type` 값만 `fasttext_candidate` → `sbert_candidate`로 바뀐다.

In [ ]:
FINAL_THETA = 0.65  # ← 위 threshold_summary 표를 보고 결정 (기본값 0.65)
TOP_PER_DIMENSION = 100

assert FINAL_THETA in aggregated_by_theta, f'{FINAL_THETA}는 THETA_GRID에 없음 — 위 셀에서 추가 sweep 필요'
base = aggregated_by_theta[FINAL_THETA]
print(f'θ={FINAL_THETA} 기준 unique 후보: {len(base):,}')
print('dimension별:', base['primary_dimension'].value_counts().to_dict())

review = pd.concat([
    base.loc[base['primary_dimension'] == dim].head(TOP_PER_DIMENSION)
    for dim in ['E', 'S', 'G']
], ignore_index=True)

review.insert(0, 'term_type', 'sbert_candidate')
review.insert(1, 'review_decision', '')
review.insert(2, 'review_reason', '')
review['theta'] = FINAL_THETA

# seed 30개는 keep 상태로 맨 위에 붙임
seed_rows = seed_dictionary[['dimension', 'seed_term']].rename(
    columns={'dimension': 'primary_dimension', 'seed_term': 'candidate'}
)
seed_rows.insert(0, 'term_type', 'seed')
seed_rows.insert(1, 'review_decision', 'keep')
seed_rows.insert(2, 'review_reason', 'provided_seed')
for col in review.columns:
    if col not in seed_rows.columns:
        seed_rows[col] = ''
seed_rows = seed_rows[review.columns]

review_template = pd.concat([seed_rows, review], ignore_index=True)

print(f'\n검토 템플릿: seed 30 + E/S/G top {TOP_PER_DIMENSION}씩 = {len(review_template)}행')
print('\n상위 8행:')
print(review_template.head(8).to_string(index=False))

print('\nE-dimension 후보 첫 10개:')
print(review_template
      .query('term_type == "sbert_candidate" and primary_dimension == "E"')
      .head(10)[['candidate', 'max_similarity', 'seed_support_count', 'supporting_seeds', 'heuristic_noise_flag']]
      .to_string(index=False))

## 9. CSV 저장 + 다운로드

3개 산출물을 `OUTPUT_DIR`에 저장한다. Colab에서는 마지막에 자동 다운로드 셀이 실행된다.

- `sbert_seed_candidates.csv` — seed별 top 300 후보 (다운스트림 LLM 필터 입력)
- `sbert_threshold_summary.csv` — θ 비교 표 (의사결정 근거)
- `expanded_dictionary_review_template.csv` — 사람/LLM 검토 템플릿 (기존 04와 동일 파일명, 다운스트림 호환)
- `sbert_similarity_histogram.png` — 유사도 분포 (참고용)

In [ ]:
seed_candidates.to_csv(SEED_CANDIDATES_CSV, index=False, encoding='utf-8-sig')
threshold_summary.to_csv(THRESHOLD_SUMMARY_CSV, index=False, encoding='utf-8-sig')
review_template.to_csv(REVIEW_TEMPLATE_CSV, index=False, encoding='utf-8-sig')

print('저장 완료:')
for p in [SEED_CANDIDATES_CSV, THRESHOLD_SUMMARY_CSV, REVIEW_TEMPLATE_CSV, HIST_PNG]:
    size_kb = p.stat().st_size / 1024 if p.exists() else 0
    print(f'  {p.name}: {size_kb:,.1f} KB')

if IN_COLAB:
    from google.colab import files
    print('\n다운로드 시작...')
    for p in [SEED_CANDIDATES_CSV, THRESHOLD_SUMMARY_CSV, REVIEW_TEMPLATE_CSV, HIST_PNG]:
        if p.exists():
            files.download(str(p))

## 다음 단계

1. `sbert_threshold_summary.csv`를 보고 `FINAL_THETA` 확정 → 셀 18 값 수정 후 재실행.
2. `expanded_dictionary_review_template.csv`를 [scripts/llm_filter_candidates.py](../scripts/llm_filter_candidates.py)로 LLM 검토 (기존 04 파이프라인 그대로). 이 노트북의 출력 스키마는 fastText 버전과 동일하므로 코드 수정 없이 사용 가능. 단, 스크립트가 입력 CSV 파일명을 하드코딩하고 있다면 `sbert_seed_candidates.csv`로 바꿔 주거나 심볼릭 링크를 만들면 된다.
3. 검토 결과(`review_decision = keep`)만 추려 expanded dictionary 확정 → seed 합산 TF-IDF 점수 ([README_작업.md](../README_작업.md) step 5/6) 생성.

### 모델 교체/비교를 하고 싶다면

이 노트북의 `MODEL_NAME`만 바꿔 같은 파이프라인을 다시 돌리면 된다:

- `BAAI/bge-m3` — KURE의 베이스, 다국어. 프리픽스 없이 그대로.
- `intfloat/multilingual-e5-large` — **`USE_QUERY_PREFIX = True`로 바꿔야 함**. e5 계열은 `"query: "` 인스트럭션이 필수.
- `BM-K/KoSimCSE-roberta` — 더 가볍지만 KURE보다 retrieval 성능은 떨어짐.

### 알려진 한계

- KURE-v1은 retrieval에 최적화된 모델이라 "의미가 가까운 단어"보다 "검색 시 함께 매치될 단어"에 점수를 더 줄 수 있다. seed가 짧은 단일 단어(예: `안전`)일 때 매우 일반적인 단어(`보호`, `관리`)도 높은 점수를 받을 수 있으니, θ 선택 시 `heuristic_noise_flag`를 함께 본다.
- 후보 풀이 DART 보고서 어휘에 갇혀 있으므로, 보고서에 한 번도 나오지 않은 ESG 용어는 expand되지 않는다. 이는 SBERT의 한계가 아니라 의도된 설계다 (실제 보고서에 없는 단어로 TF-IDF 점수를 못 매기므로).